# Cross-resonance calibration workflow (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

This notebook sketches a practical CR bring-up for one coupled qubit pair:
prepare the single-qubit pulse set, obtain CR parameters, calibrate a
`ZX90` gate, inspect the resulting schedule, and validate it with Bell
measurements and interleaved RB.

## 1. Create an `Experiment`

Pick a known coupled pair for your device. The example below uses the
first two qubits from the selected subset, but you should replace them
with a pair that already has a calibrated CR path in your system.

In [1]:
import numpy as np

from IPython.display import display

import qubex_emulator as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [2]:
if len(exp.qubit_labels) < 2:
    raise ValueError("Select at least two coupled qubits for CR calibration.")

cr_control, cr_target = exp.qubit_labels[:2]
cr_label = f"{cr_control}-{cr_target}"

print("control qubit:", cr_control)
print("target qubit:", cr_target)
print("pair label:", cr_label)

control qubit: Q00
target qubit: Q01
pair label: Q00-Q01


## 2. Connect to the setup

Connect before you prepare the single-qubit prerequisites or run any CR measurements.

In [3]:
exp.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-cr-cr', drag_hpi_puls

## 3. Prepare the single-qubit prerequisites

Measure the baseline Rabi response, calibrate the single-qubit pulses, and build the readout classifier before you move to CR calibration.

In [4]:
result_rabi = exp.obtain_rabi_params()
result_hpi = exp.calibrate_hpi_pulse()
result_pi = exp.calibrate_pi_pulse()
result_drag_hpi = exp.calibrate_drag_hpi_pulse()
result_drag_pi = exp.calibrate_drag_pi_pulse()
result_classifier = exp.build_classifier()

## 4. Obtain the CR interaction parameters

Use the CR parameter scan to estimate a usable operating point for the entangling drive.

In [5]:
result_cr = exp.obtain_cr_params(cr_control, cr_target)

## 5. Calibrate the `ZX90` gate

Run the `ZX90` calibration once the CR interaction parameters are in place.

In [6]:
result_cr_calib = exp.calibrate_zx90(
    control_qubit=cr_control,
    target_qubit=cr_target,
)

## 6. Inspect the cached `ZX90` schedule

Plot the calibrated schedule so you can inspect the pulse structure directly.

In [7]:
zx90 = exp.zx90(cr_control, cr_target)

display(zx90.plot())

None

## 7. Repeat the calibrated `ZX90` sequence

Use a repeated-sequence check to confirm that the calibrated gate behaves sensibly.

In [8]:
repeat_result = exp.repeat_sequence(zx90)

## 8. Run pulse tomography with the control qubit in `|0>`

Reconstruct the repeated `ZX90` operation with the control qubit initialized in `|0>`.

In [9]:
pulse_tomography_0 = exp.pulse_tomography(zx90, initial_state={cr_control: "0"})

## 9. Run pulse tomography with the control qubit in `|1>`

Repeat the same tomography measurement with the control qubit initialized in `|1>`.

In [10]:
pulse_tomography_1 = exp.pulse_tomography(zx90, initial_state={cr_control: "1"})

## 10. Measure the Bell state

Use a Bell-state measurement as a direct entangling-gate validation step.

In [11]:
result_bell = exp.measure_bell_state(cr_control, cr_target)

result_bell_tomography = exp.bell_state_tomography(cr_control, cr_target)

## 11. Run interleaved randomized benchmarking

Estimate the `ZX90` gate error with interleaved RB.

In [12]:
result_irb = exp.interleaved_randomized_benchmarking(
    cr_label,
    interleaved_clifford="ZX90",
    n_trials=30,
)

## 12. Save the calibration note

Save the final CR calibration once the validation steps look acceptable.

In [13]:
exp.calib_note.save()